# Function 1 - Logistic Regression with Hyperparameter Tuning

This notebook:
1. Fits a logistic regression to Function 1 data
2. Tunes hyperparameters to find the model with lowest MSE
3. Uses the best model to predict the next point closest to the decision boundary

## Step 0: Import Packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, log_loss

import sys
sys.path.append('..')

import importlib
import scripts.utils.data_utils as data_utils
importlib.reload(data_utils)

from scripts.utils.data_utils import (
    load_initial_data,
    load_week1_data,
    load_week2_data,
    load_week3_data,
    plot_2d_scatter_with_distribution
)

from scripts.logistic_regression import SimpleLogisticRegression, make_binary_labels

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Step 1: Load Data for Function 1

In [ ]:
# Load all data
base_dir = Path('../data/initial_data')
week1_dir = Path('../data/week1')
week2_dir = Path('../data/week2')
week3_dir = Path('../data/week3')

initial_data = load_initial_data(base_dir)
week1_data = load_week1_data(week1_dir)
week2_data = load_week2_data(week2_dir)
week3_data = load_week3_data(week3_dir)

print(f"Loaded data for {len(initial_data)} functions")

## Step 2: Prepare Function 1 Data

In [ ]:
function_name = 'function_1'

# Get original inputs and outputs
original_inputs = initial_data[function_name]['inputs']
original_outputs = initial_data[function_name]['outputs']

# Get week data
week1_input = week1_data[function_name]['input']
week1_output = week1_data[function_name]['output']
week2_input = week2_data[function_name]['input']
week2_output = week2_data[function_name]['output']
week3_input = week3_data[function_name]['input']
week3_output = week3_data[function_name]['output']

# Combine all data
X_all = np.vstack([
    original_inputs,
    week1_input.reshape(1, -1),
    week2_input.reshape(1, -1),
    week3_input.reshape(1, -1)
])

y_raw = np.concatenate([
    original_outputs,
    np.array([week1_output]),
    np.array([week2_output]),
    np.array([week3_output])
])

print("="*60)
print(f"FUNCTION 1 - COMBINED DATA")
print("="*60)
print(f"Combined inputs shape: {X_all.shape}")
print(f"Combined outputs shape: {y_raw.shape}")
print(f"\nInput range: [{X_all.min():.3f}, {X_all.max():.3f}]")
print(f"Output range: [{y_raw.min():.6e}, {y_raw.max():.6e}]")
print("="*60)

## Step 3: Visualize the Data

In [ ]:
# Visualize with all weeks
plot_2d_scatter_with_distribution(
    original_inputs, original_outputs, 
    week3_input, week3_output, 
    function_name,
    prior_input=week2_input,
    prior_output=week2_output,
    prior_point_label="Week 2 point",
    new_point_label="Week 3 point",
    connect_points=True,
    connect_color="tab:orange"
)

## Step 4: Hyperparameter Tuning for Logistic Regression

We'll tune:
- `threshold_percentile`: For creating binary labels (50, 60, 70, 75, 80, 90)
- `penalty`: Regularization type (None, 'l2')
- `C`: Inverse regularization strength (0.01, 0.1, 1.0, 10.0, 100.0)
- `max_iter`: Maximum iterations (1000, 2000, 5000)

We'll use Mean Squared Error (MSE) on the predicted probabilities vs. binary targets.

In [ ]:
# Define hyperparameter grid
threshold_percentiles = [50, 60, 70, 75, 80, 90]
penalties = [None, 'l2']
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
max_iters = [1000, 2000, 5000]

# Store results
results = []

print("Starting hyperparameter tuning...")
print(f"Total combinations: {len(threshold_percentiles) * len(penalties) * len(C_values) * len(max_iters)}")
print("="*80)

for threshold_percentile in threshold_percentiles:
    # Create binary labels
    Y, threshold_used = make_binary_labels(y_raw, threshold_percentile=threshold_percentile)
    
    # Check if we have both classes
    if np.unique(Y).size < 2:
        print(f"Skipping threshold_percentile={threshold_percentile} (produces single class)")
        continue
    
    class_balance = f"{(Y == 0).sum()}/{(Y == 1).sum()}"
    
    for penalty in penalties:
        for max_iter in max_iters:
            # Handle penalty=None case (no C parameter)
            if penalty is None:
                C_list = [1.0]  # C is ignored when penalty=None
            else:
                C_list = C_values
            
            for C in C_list:
                try:
                    # Create and train model
                    if penalty is None:
                        log_reg = SimpleLogisticRegression(
                            penalty=None,
                            random_state=RANDOM_STATE,
                            max_iter=max_iter,
                            solver='lbfgs'
                        )
                    else:
                        # For l2 penalty, we need to modify the class to accept C
                        from sklearn.linear_model import LogisticRegression
                        from sklearn.preprocessing import StandardScaler
                        
                        log_reg = SimpleLogisticRegression(
                            penalty=penalty,
                            random_state=RANDOM_STATE,
                            max_iter=max_iter,
                            solver='lbfgs'
                        )
                        # Override the model with C parameter
                        log_reg.model = LogisticRegression(
                            penalty=penalty,
                            C=C,
                            random_state=RANDOM_STATE,
                            max_iter=max_iter,
                            solver='lbfgs'
                        )
                    
                    # Fit and predict
                    X_scaled = log_reg.fit_scaler(X_all)
                    log_reg.fit(X_scaled, Y)
                    Y_pred = log_reg.predict(X_scaled)
                    Y_pred_proba = log_reg.predict_proba(X_scaled)[:, 1]
                    
                    # Calculate metrics
                    accuracy = log_reg.accuracy(Y, Y_pred)
                    mse = mean_squared_error(Y, Y_pred_proba)
                    logloss = log_loss(Y, Y_pred_proba)
                    
                    # Store result
                    results.append({
                        'threshold_percentile': threshold_percentile,
                        'threshold_value': threshold_used,
                        'class_balance': class_balance,
                        'penalty': penalty if penalty else 'None',
                        'C': C if penalty else np.nan,
                        'max_iter': max_iter,
                        'accuracy': accuracy,
                        'mse': mse,
                        'log_loss': logloss
                    })
                    
                except Exception as e:
                    print(f"Error with params (threshold={threshold_percentile}, penalty={penalty}, C={C}, max_iter={max_iter}): {str(e)}")
                    continue

# Convert to DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*80)
print(f"Completed! Tested {len(results_df)} configurations")
print("="*80)

## Step 5: Analyze Results

In [ ]:
# Sort by MSE (lower is better)
results_sorted = results_df.sort_values('mse')

print("\n" + "="*80)
print("TOP 10 MODELS BY MSE")
print("="*80)
print(results_sorted.head(10).to_string(index=False))

print("\n" + "="*80)
print("TOP 10 MODELS BY ACCURACY")
print("="*80)
results_by_acc = results_df.sort_values('accuracy', ascending=False)
print(results_by_acc.head(10).to_string(index=False))

print("\n" + "="*80)
print("TOP 10 MODELS BY LOG LOSS")
print("="*80)
results_by_loss = results_df.sort_values('log_loss')
print(results_by_loss.head(10).to_string(index=False))

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MSE vs threshold percentile
axes[0, 0].scatter(results_df['threshold_percentile'], results_df['mse'], alpha=0.6)
axes[0, 0].set_xlabel('Threshold Percentile')
axes[0, 0].set_ylabel('MSE')
axes[0, 0].set_title('MSE vs Threshold Percentile')
axes[0, 0].grid(True, alpha=0.3)

# Accuracy vs threshold percentile
axes[0, 1].scatter(results_df['threshold_percentile'], results_df['accuracy'], alpha=0.6)
axes[0, 1].set_xlabel('Threshold Percentile')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy vs Threshold Percentile')
axes[0, 1].grid(True, alpha=0.3)

# MSE by penalty type
penalty_groups = results_df.groupby('penalty')['mse'].apply(list)
axes[1, 0].boxplot([penalty_groups[p] for p in penalty_groups.index], 
                    labels=penalty_groups.index)
axes[1, 0].set_ylabel('MSE')
axes[1, 0].set_xlabel('Penalty')
axes[1, 0].set_title('MSE Distribution by Penalty Type')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Log Loss vs MSE
axes[1, 1].scatter(results_df['mse'], results_df['log_loss'], alpha=0.6)
axes[1, 1].set_xlabel('MSE')
axes[1, 1].set_ylabel('Log Loss')
axes[1, 1].set_title('Log Loss vs MSE')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Train Best Model

In [ ]:
# Get best parameters
best_params = results_sorted.iloc[0]

print("="*80)
print("BEST MODEL PARAMETERS (by MSE)")
print("="*80)
for key, value in best_params.items():
    print(f"{key:25s}: {value}")
print("="*80)

# Recreate best model
Y_best, threshold_best = make_binary_labels(
    y_raw, 
    threshold_percentile=int(best_params['threshold_percentile'])
)

if best_params['penalty'] == 'None':
    best_log_reg = SimpleLogisticRegression(
        penalty=None,
        random_state=RANDOM_STATE,
        max_iter=int(best_params['max_iter']),
        solver='lbfgs'
    )
else:
    from sklearn.linear_model import LogisticRegression
    best_log_reg = SimpleLogisticRegression(
        penalty=best_params['penalty'],
        random_state=RANDOM_STATE,
        max_iter=int(best_params['max_iter']),
        solver='lbfgs'
    )
    best_log_reg.model = LogisticRegression(
        penalty=best_params['penalty'],
        C=best_params['C'],
        random_state=RANDOM_STATE,
        max_iter=int(best_params['max_iter']),
        solver='lbfgs'
    )

# Fit best model
X_scaled_best = best_log_reg.fit_scaler(X_all)
best_log_reg.fit(X_scaled_best, Y_best)

print("\n✓ Best model trained successfully!")

## Step 7: Predict Next Point Closest to Decision Boundary

We'll use three methods to find the point closest to the decision boundary (probability ≈ 0.5):
1. Grid search
2. Random search
3. Optimization

In [ ]:
# Define input space bounds (assuming [0, 1] for both dimensions)
bounds = [[0.0, 1.0], [0.0, 1.0]]

print("="*80)
print("FINDING NEXT POINT CLOSEST TO DECISION BOUNDARY")
print("="*80)

# Method 1: Grid search
print("\n1. Grid Search Method:")
next_point_grid, uncertainty_grid, prob_grid = best_log_reg.find_next_point_near_boundary(
    bounds=bounds,
    method='grid_search',
    n_candidates=10000,
    random_state=RANDOM_STATE
)
print(f"   Next point: {next_point_grid}")
print(f"   Probability: {prob_grid:.6f}")
print(f"   Distance from boundary: {uncertainty_grid:.6f}")

# Method 2: Random search
print("\n2. Random Search Method:")
next_point_random, uncertainty_random, prob_random = best_log_reg.find_next_point_near_boundary(
    bounds=bounds,
    method='random',
    n_candidates=10000,
    random_state=RANDOM_STATE
)
print(f"   Next point: {next_point_random}")
print(f"   Probability: {prob_random:.6f}")
print(f"   Distance from boundary: {uncertainty_random:.6f}")

# Method 3: Optimization
print("\n3. Optimization Method (most accurate):")
next_point_opt, uncertainty_opt, prob_opt = best_log_reg.find_next_point_near_boundary(
    bounds=bounds,
    method='optimize',
    random_state=RANDOM_STATE
)
print(f"   Next point: {next_point_opt}")
print(f"   Probability: {prob_opt:.6f}")
print(f"   Distance from boundary: {uncertainty_opt:.6f}")

print("\n" + "="*80)
print(f"RECOMMENDED NEXT POINT (using optimization): {next_point_opt}")
print("="*80)

## Step 8: Visualize Decision Boundary and Next Point

In [ ]:
# Create a grid for visualization
n_grid = 100
x1_range = np.linspace(0, 1, n_grid)
x2_range = np.linspace(0, 1, n_grid)
X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)
grid_points = np.column_stack([X1_grid.ravel(), X2_grid.ravel()])

# Get predictions for grid
grid_scaled = best_log_reg.transform(grid_points)
probs_grid = best_log_reg.predict_proba(grid_scaled)[:, 1].reshape(n_grid, n_grid)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Decision boundary with probability contours
ax = axes[0]
contour = ax.contourf(X1_grid, X2_grid, probs_grid, levels=20, cmap='RdYlBu_r', alpha=0.7)
ax.contour(X1_grid, X2_grid, probs_grid, levels=[0.5], colors='black', linewidths=2, linestyles='--')

# Plot training points
scatter = ax.scatter(X_all[:, 0], X_all[:, 1], c=Y_best, cmap='RdYlBu_r', 
                    s=100, edgecolors='black', linewidths=1.5, zorder=5)

# Plot suggested next points
ax.scatter(next_point_opt[0], next_point_opt[1], c='lime', marker='*', 
          s=500, edgecolors='black', linewidths=2, zorder=10, label='Next point (optimize)')
ax.scatter(next_point_grid[0], next_point_grid[1], c='yellow', marker='D', 
          s=200, edgecolors='black', linewidths=1.5, zorder=9, label='Next point (grid)')

ax.set_xlabel('x1', fontsize=12)
ax.set_ylabel('x2', fontsize=12)
ax.set_title('Decision Boundary and Suggested Next Point', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.colorbar(contour, ax=ax, label='P(Class 1)')

# Right plot: Uncertainty map
ax = axes[1]
uncertainties_grid = np.abs(probs_grid - 0.5)
contour2 = ax.contourf(X1_grid, X2_grid, uncertainties_grid, levels=20, cmap='viridis', alpha=0.7)

# Plot training points
ax.scatter(X_all[:, 0], X_all[:, 1], c='white', s=100, edgecolors='black', 
          linewidths=1.5, zorder=5, label='Training data')

# Plot suggested next point
ax.scatter(next_point_opt[0], next_point_opt[1], c='lime', marker='*', 
          s=500, edgecolors='black', linewidths=2, zorder=10, label='Next point (optimize)')

ax.set_xlabel('x1', fontsize=12)
ax.set_ylabel('x2', fontsize=12)
ax.set_title('Uncertainty Map (closer to 0 = more uncertain)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.colorbar(contour2, ax=ax, label='|P - 0.5|')

plt.tight_layout()
plt.show()

In [ ]:
print("="*80)
print("SUMMARY - FUNCTION 1 LOGISTIC REGRESSION")
print("="*80)
print(f"\nData Summary:")
print(f"  - Total samples: {len(X_all)}")
print(f"  - Input dimensions: {X_all.shape[1]}")
print(f"  - Input space: [0, 1] x [0, 1]")
print(f"  - Output range: [{y_raw.min():.6e}, {y_raw.max():.6e}]")

print(f"\nBest Model:")
print(f"  - Threshold percentile: {best_params['threshold_percentile']}")
print(f"  - Threshold value: {best_params['threshold_value']:.6e}")
print(f"  - Class balance (0/1): {best_params['class_balance']}")
print(f"  - Penalty: {best_params['penalty']}")
if best_params['penalty'] != 'None':
    print(f"  - C: {best_params['C']}")
print(f"  - Max iterations: {best_params['max_iter']}")
print(f"  - MSE: {best_params['mse']:.6f}")
print(f"  - Accuracy: {best_params['accuracy']:.4f}")
print(f"  - Log Loss: {best_params['log_loss']:.6f}")

print(f"\nRecommended Next Point:")
print(f"  - Coordinates: {next_point_opt}")
print(f"  - Predicted probability: {prob_opt:.6f}")
print(f"  - Distance from boundary: {uncertainty_opt:.6f}")

print(f"\nInterpretation:")
print(f"  This point is closest to the decision boundary (P ≈ 0.5), making it")
print(f"  the most uncertain point for the model. Evaluating this point will")
print(f"  provide maximum information to refine the decision boundary.")
print("="*80)

## Step 10: Save Results

In [ ]:
# Save all results to CSV
results_df.to_csv('function1_logistic_regression_results.csv', index=False)
print("✓ Results saved to 'function1_logistic_regression_results.csv'")

# Save recommended next point
with open('function1_next_point.txt', 'w') as f:
    f.write(f"Recommended next point for Function 1:\n")
    f.write(f"x1 = {next_point_opt[0]:.10f}\n")
    f.write(f"x2 = {next_point_opt[1]:.10f}\n")
    f.write(f"\nPredicted probability: {prob_opt:.6f}\n")
    f.write(f"Distance from boundary: {uncertainty_opt:.6f}\n")
print("✓ Next point saved to 'function1_next_point.txt'")